In [1]:
%load_ext autoreload
%autoreload 2

import sys, os

import pandas as pd
from pathlib import Path
from src.inference import ASRInference
from src.metrics import cer


In [11]:
CKPT = Path("logs/unsplittable-v2/version_0/checkpoints/best_epoch17_harmonic_3.24.ckpt")
asr = ASRInference(CKPT, device="cuda")
print("Model loaded.")


Model loaded.


/home/gorlov/asr_env/lib/python3.12/site-packages/torchaudio/functional/functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (80) may be set too high. Or, the value for `n_freqs` (129) may be set too low.
  warnings.warn(


In [12]:
dev_df = pd.read_csv("data/dev.csv")
paths = [Path("data") / fn for fn in dev_df["filename"]]

preds = asr.transcribe(paths)

errors = [cer(p, str(t)) for p, t in zip(preds, dev_df["transcription"])]
print(f"Mean CER on dev: {sum(errors)/len(errors):.4f}")
for p, t in list(zip(preds, dev_df["transcription"]))[:5]:
    print(f"  pred={p!r:10s}  truth={t}")


Mean CER on dev: 0.0412
  pred='849905'    truth=849905
  pred='967653'    truth=967653
  pred='427524'    truth=427524
  pred='93752'     truth=93752
  pred='43355'     truth=43355


In [8]:
from pathlib import Path
import pandas as pd

print("test.csv:", Path("data/test.csv").exists())
print("sample_submission.csv:", Path("data/sample_submission.csv").exists())
print("test/ files:", len(list(Path("data/test").glob("*"))))

test_df = pd.read_csv("data/test.csv")
print(f"\ntest.csv: {len(test_df)} rows")
print(test_df.head())
print("\ncolumns:", list(test_df.columns))


test.csv: True
sample_submission.csv: True
test/ files: 2582

test.csv: 2582 rows
              filename  ext  samplerate
0  test/d2440788a9.mp3  mp3       16000
1  test/e247dbf761.mp3  mp3       16000
2  test/071f4a5be7.mp3  mp3       16000
3  test/798bd15db7.mp3  mp3       16000
4  test/58c0464ad5.mp3  mp3       16000

columns: ['filename', 'ext', 'samplerate']


In [9]:
first_path = Path("data") / test_df.iloc[0]["filename"]
print(f"First file: {first_path}  exists={first_path.exists()}")
pred = asr.transcribe([first_path])
print(f"Prediction: {pred[0]}")


First file: data/test/d2440788a9.mp3  exists=True
Prediction: 461694


In [17]:
paths = [Path("data") / fn for fn in test_df["filename"]]
preds = asr.transcribe(paths)   # ~3–5 min for ~1000 files

submission = pd.DataFrame({
    "filename":      test_df["filename"],
    "transcription": preds,
})
submission.to_csv("submission.csv", index=False)
print(f"Wrote {len(submission)} rows")
print(submission.head())


Wrote 2582 rows
              filename transcription
0  test/d2440788a9.mp3        461694
1  test/e247dbf761.mp3        207723
2  test/071f4a5be7.mp3         79187
3  test/798bd15db7.mp3         64048
4  test/58c0464ad5.mp3        861168


In [2]:
CKPT = Path("logs/conformer-v2/version_0/checkpoints/epoch=23-val_harmonic_cer=0.0094.ckpt")
asr = ASRInference(CKPT, device="cuda")
print("Model loaded")


Model loaded


/home/gorlov/asr_env/lib/python3.12/site-packages/torchaudio/functional/functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (80) may be set too high. Or, the value for `n_freqs` (129) may be set too low.
  warnings.warn(


In [3]:
import pandas as pd
from src.metrics import cer

dev_df = pd.read_csv("data/dev.csv")
dev_paths = [Path("data") / f for f in dev_df["filename"].tolist()]
truths = dev_df["transcription"].astype(str).tolist()

preds = asr.transcribe(dev_paths)

cers = [cer(p, t) for p, t in zip(preds, truths)]
print(f"Mean dev CER: {sum(cers)/len(cers):.4f}")
for p, t in zip(preds[:5], truths[:5]):
    print(f"pred={p!r:>12}  truth={t}")


Mean dev CER: 0.0361
pred=    '849905'  truth=849905
pred=    '247053'  truth=967653
pred=    '427524'  truth=427524
pred=     '93752'  truth=93752
pred=     '43355'  truth=43355


In [4]:
test_df = pd.read_csv("data/test.csv")
test_paths = [Path("data") / f for f in test_df["filename"].tolist()]

predictions = asr.transcribe(test_paths)

submission = pd.DataFrame({
    "filename": test_df["filename"],
    "transcription": predictions,
})
submission.to_csv("submission_v2.csv", index=False)
print(f"Wrote {len(submission)} rows")
submission.head()


Wrote 2582 rows


,filename,transcription
0,test/d2440788a9.mp3,461694
1,test/e247dbf761.mp3,207723
2,test/071f4a5be7.mp3,79107
3,test/798bd15db7.mp3,64048
4,test/58c0464ad5.mp3,861162


In [5]:
print("Module:", asr.model.__class__.__module__)
print("Params:", sum(p.numel() for p in asr.model.parameters()) / 1e6, "M")


Module: src.models.conformer
Params: 2.507078 M


In [12]:
import pandas as pd
from pathlib import Path

test_df = pd.read_csv("data/test.csv")
test_df["transcription"] = "0"      # dummy, model output is what we use
test_df["spk_id"] = "unk"           # ASRDataset also reads this column
test_df.to_csv("data/test_with_dummy.csv", index=False)


In [ ]:
from torch.utils.data import DataLoader
from src.dataset import ASRDataset
from src.metrics import ctc_decode
import torch

test_ds = ASRDataset(
    csv_path="data/test_with_dummy.csv",
    root_dir=Path("data"),
    normalizer=asr.normalizer,
    augmenter=None,
    sample_rate=16000, n_fft=256, hop_length=160, n_mels=80,
)

def collate_fn(batch):
    mels, labels, spk_ids = zip(*batch)
    mel_lengths = torch.tensor([m.shape[0] for m in mels])
    mels = torch.nn.utils.rnn.pad_sequence(mels, batch_first=True)
    return mels, mel_lengths

test_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                        collate_fn=collate_fn, num_workers=4)

asr.model.eval()
predictions = []
with torch.no_grad():
    for mels, lengths in test_loader:
        mels = mels.to("cuda")
        logits = asr.model(mels, lengths.to("cuda"))
        preds = torch.argmax(logits, dim=-1)
        T_out = logits.size(1)
        out_lengths = (((lengths + 1) // 2 + 1) // 2).clamp(max=T_out)
        for i in range(preds.size(0)):
            toks = ctc_decode(preds[i, :out_lengths[i]].cpu().numpy())
            predictions.append(asr.normalizer.tokens2num(toks))

test_df = pd.read_csv("data/test_with_dummy.csv")
submission = pd.DataFrame({
    "filename": test_df["filename"],
    "transcription": predictions,
})
submission.to_csv("submission_v3.csv", index=False)
print(f"Wrote {len(submission)} rows")


/home/gorlov/asr_env/lib/python3.12/site-packages/torchaudio/functional/functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (80) may be set too high. Or, the value for `n_freqs` (129) may be set too low.
  warnings.warn(
Loading audio files: 100%|██████████| 2582/2582 [00:25<00:00, 101.93it/s]


KeyError: 'transcription'